# 06 · Evaluate and analyse  (steps 10, 13, 14; CPU session, no GPU quota)
Attach the finished runs (Dataset **pedger-runs**). Final numbers: test split, models trained on trainval.
Ablations / dev runs: val split.

In [ ]:
# --- setup: code + upstream PEdger -------------------------------------------
import os, subprocess, sys
REPO_URL = ""          # e.g. "https://github.com/<you>/pedger-mtkd.git" (or attach the code as a Kaggle Dataset)
CODE_DATASET = "/kaggle/input/pedger-mtkd-code"
W = "/kaggle/working"
REPO = f"{W}/pedger-mtkd"
if not os.path.exists(REPO):
    if REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
    else:
        subprocess.run(["cp", "-r", CODE_DATASET, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
!python tools/fetch_upstream.py
!pip install -q -r requirements.txt

In [ ]:
# --- inputs (attach these Kaggle Datasets; names are suggestions) --------------
RAW_BSDS = "/kaggle/input/bsds500-raw/BSR/BSDS500/data"     # original Berkeley release
DATA = "/kaggle/input/pedger-data"                           # output of 01 (array stores), saved as a Dataset
CACHE = "/kaggle/input/pedger-teachers"                      # output of 03 (teacher cache), saved as a Dataset
RUNS = "/kaggle/working/runs"
import os
if not os.path.exists(DATA): DATA = "/kaggle/working/data_store"
if not os.path.exists(CACHE): CACHE = "/kaggle/working/teacher_cache"
RUNS = "/kaggle/input/pedger-runs"
OUT = "/kaggle/working/eval_runs"
!mkdir -p {OUT} && cp -r {RUNS}/* {OUT}/

In [ ]:
import glob, json
for run in sorted(glob.glob(f"{OUT}/*")):
    if not os.path.exists(f"{run}/config.yaml"): continue
    split = "val" if "split: train\n" in open(f"{run}/config.yaml").read() else "test"
    !python eval/run_bsds_eval.py --run {run} --store {DATA}/bsds --split {split} --raw-root {RAW_BSDS} --nproc 4 --device cpu
    # multi-scale for the main rows only:
    # !python eval/run_bsds_eval.py --run {run} --ms --store {DATA}/bsds --split {split} --raw-root {RAW_BSDS} --nproc 4 --device cpu

In [ ]:
!python results/make_tables.py --scores "{OUT}/*/scores_*.json" --out /kaggle/working/tables.md

In [ ]:
# Agreement-bin analysis (step 13): baseline vs ours, seed 0, test split
!python eval/agreement_bins.py --store {DATA}/bsds --split test --raw-root {RAW_BSDS}  --run PEdger={OUT}/baseline_s0/scores_test_ns_m.json --run Ours={OUT}/A_agree_s0/scores_test_ns_m.json  --run A_uniform={OUT}/A_uniform_s0/scores_test_ns_m.json --out /kaggle/working/agreement_bins.json

In [ ]:
# Efficiency and training cost (step 14). Run once on a GPU session for T4 FPS; CPU FPS works here.
TEACHER_HOURS = 0.0   # from notebook 03
!python bench/speed_flops.py --size 320 480 --runs "{OUT}/*" --teacher-hours {TEACHER_HOURS} --out /kaggle/working/efficiency.json